In [ ]:
# 1. Montar o Google Drive no Colab
from google.colab import drive
drive.mount('/content/drive')

# 2. Ler o arquivo CSV usando o caminho do Drive
import pandas as pd

# Sempre avalair o nme do arquivo e verificar se vamos ler a pasta RAW ou Bronze
caminho = "/content/drive/MyDrive/mini-airflow/03_processado_bronze/20260411_131424_data_frame_python.csv"

df_raw = pd.read_csv(caminho)

# 3. Visualizar o DataFrame
df_raw.head()


ValueError: mount failed

In [ ]:
total_linhas = df_raw.shape[0]
total_linhas

6237

In [ ]:
# 1. Montar o Google Drive no Colab
from google.colab import drive
drive.mount('/content/drive')

# 2. Ler o arquivo PARQUET usando o caminho do Drive
import pandas as pd

caminho = "/content/drive/MyDrive/mini-airflow/05_business_silver/tb_df_py.parquet"

df_silver = pd.read_parquet(caminho)

# 3. Visualizar o DataFrame
df_silver.head()

Mounted at /content/drive


,id_registro,id_pessoa,id_produto,tp_classe,nr_valor,dt_criacao,ano,mes
0,20260410_221417_000001,99,3,2,246.98,2026-04-10 22:14:17.839409,2026,4
1,20260410_221417_000002,83,2,3,136.34,2026-04-10 22:14:17.839416,2026,4
2,20260410_221417_000003,20,3,1,271.31,2026-04-10 22:14:17.839420,2026,4
3,20260410_221417_000004,65,4,3,431.41,2026-04-10 22:14:17.839423,2026,4
4,20260410_221417_000005,22,3,2,109.68,2026-04-10 22:14:17.839427,2026,4


In [ ]:
silver = (
    df_silver.groupby(["id_pessoa"])
      .agg(total_valor=("nr_valor", "sum"))
      .reset_index()
)

silver

,id_pessoa,total_valor
0,1,6039185.03
1,10,1753679.83
2,100,886446.53
3,11,1653368.58
4,12,1615499.54
...,...,...
95,95,875436.59
96,96,887422.26
97,97,896734.56
98,98,878492.34


In [ ]:
total_linhas = df_silver.shape[0]
total_linhas


408964

In [ ]:
# 1. Montar o Google Drive no Colab
from google.colab import drive
drive.mount('/content/drive')

# 2. Ler o arquivo PARQUET usando o caminho do Drive
import pandas as pd

caminho = "/content/drive/MyDrive/mini-airflow/06_analitico_gold/tb_fato.parquet"

df_gold = pd.read_parquet(caminho)

# 3. Visualizar o DataFrame
df_gold.head()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


,id_registro,id_pessoa,id_produto,tp_classe,nr_valor,dt_criacao,ano,mes,id_classe,nome_classe,fator_classe,valor_ponderado
0,20260414_161446_000001,1,2,1,400.75,2026-04-14 16:14:46.162756,2026,4,1,Online,0.35,140.2625
1,20260414_161446_000002,38,2,2,158.31,2026-04-14 16:14:46.162787,2026,4,2,Streaming,0.15,23.7465
2,20260414_161446_000003,16,4,1,361.52,2026-04-14 16:14:46.162798,2026,4,1,Online,0.35,126.5320
3,20260414_161446_000004,10,1,1,435.14,2026-04-14 16:14:46.162806,2026,4,1,Online,0.35,152.2990
4,20260414_161446_000005,40,2,1,258.59,2026-04-14 16:14:46.162814,2026,4,1,Online,0.35,90.5065


In [ ]:
total_linhas = df_gold.shape[0]
total_linhas

39654

In [ ]:
print(df_gold.dtypes)

id_registro                object
id_pessoa                  object
id_produto                 object
tp_classe                  object
nr_valor                  float64
dt_criacao         datetime64[us]
ano                         int32
mes                         int32
id_classe                  object
nome_classe                object
fator_classe              float64
valor_ponderado           float64
dtype: object


In [7]:
# ==============================================================================
# SCRIPT DE AUDITORIA E VALIDAÇÃO DE DADOS (LAKEHOUSE)
# Objetivo: Garantir a integridade entre os arquivos físicos e o Dashboard
# Ler METADADOS PARQUET
# ==============================================================================
import pyarrow.parquet as pq

# 1. Montar o Google Drive no Colab
from google.colab import drive
drive.mount('/content/drive')

# 2. Ler o arquivo PARQUET usando o caminho do Drive
import pandas as pd

# Caminho do seu arquivo (substitua pelo seu)
caminho = "/content/drive/MyDrive/mini-airflow/06_analitico_gold/tb_fato.parquet"

# Lendo apenas os metadados (sem carregar os dados na memória)
metadata = pq.read_metadata(caminho)

print(f"Número de Colunas: {metadata.num_columns}")
print(f"Número de Linhas: {metadata.num_rows}")
print(f"Número de Grupos de Linhas (Row Groups): {metadata.num_row_groups}")

# Ver o esquema detalhado
print("\n--- Esquema ---")
print(metadata.schema)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Número de Colunas: 12
Número de Linhas: 124872
Número de Grupos de Linhas (Row Groups): 1

--- Esquema ---
required group field_id=-1 schema {
  optional binary field_id=-1 id_registro (String);
  optional binary field_id=-1 id_pessoa (String);
  optional binary field_id=-1 id_produto (String);
  optional binary field_id=-1 tp_classe (String);
  optional double field_id=-1 nr_valor;
  optional int64 field_id=-1 dt_criacao (Timestamp(isAdjustedToUTC=false, timeUnit=microseconds, is_from_converted_type=false, force_set_converted_type=false));
  optional int32 field_id=-1 ano;
  optional int32 field_id=-1 mes;
  optional binary field_id=-1 id_classe (String);
  optional binary field_id=-1 nome_classe (String);
  optional double field_id=-1 fator_classe;
  optional double field_id=-1 valor_ponderado;
}



In [ ]:
# Validar Google Storage
from google.colab import auth
from google.cloud import storage
import pandas as pd
import io

auth.authenticate_user() # Vai pedir login do Google

client = storage.Client()
bucket = client.bucket("mini-airflow-gold-data")
blob = bucket.blob("tb_fato.parquet")

# Lê o conteúdo do Parquet direto da memória
content = blob.download_as_bytes()
df_nuvem = pd.read_parquet(io.BytesIO(content))

print(f"Quantidade de linhas no Storage: {len(df_nuvem)}")
print(df_nuvem['dt_criacao'].dtype) # Valida se o tipo veio correto

Quantidade de linhas no Storage: 86991
datetime64[us]


In [5]:
# ==============================================================================
# SCRIPT DE AUDITORIA E VALIDAÇÃO DE DADOS (LAKEHOUSE)
# Objetivo: Garantir a integridade entre os arquivos físicos e o Dashboard
# ==============================================================================

import pandas as pd
from google.colab import drive

# 1. Montar o Google Drive
drive.mount('/content/drive')

# 2. Caminho do seu CSV (Manifesto)
caminho = "/content/drive/MyDrive/mini-airflow/03_processado_bronze/manifesto_ingestao.csv"

# 3. Ler o CSV usando o delimitador correto (que vimos ser o ponto e vírgula)
df_manifesto = pd.read_csv(caminho, sep=';')

# 4. Ver as métricas de auditoria que você quer bater no Looker
print(f"Número de Colunas: {len(df_manifesto.columns)}")
print(f"Número de Linhas (Total de Arquivos): {len(df_manifesto)}")
print(f"Soma Total de Registros (O que deve estar no Looker): {df_manifesto['quantidade_linhas'].sum()}")

# 5. Ver o esquema (Tipos de dados)
print("\n--- Esquema ---")
print(df_manifesto.dtypes)

# Ver os últimos lotes processados
print("\n--- Últimas Cargas ---")
print(df_manifesto.tail())


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Número de Colunas: 4
Número de Linhas (Total de Arquivos): 17
Soma Total de Registros (O que deve estar no Looker): 124872

--- Esquema ---
data_processamento    object
arquivo_origem        object
quantidade_linhas      int64
status                object
dtype: object

--- Últimas Cargas ---
     data_processamento                         arquivo_origem  \
12  2026-04-16 15:07:35  20260416_150733_data_frame_python.csv   
13  2026-04-16 16:00:22  20260416_160020_data_frame_python.csv   
14  2026-04-16 18:03:45  20260416_180343_data_frame_python.csv   
15  2026-04-16 19:17:23  20260416_191721_data_frame_python.csv   
16  2026-04-16 19:26:44  20260416_192642_data_frame_python.csv   

    quantidade_linhas   status  
12               6656  SUCESSO  
13               7033  SUCESSO  
14               8036  SUCESSO  
15               8709  SUCESSO  
16             